# 13 — Produtos TOTVS e termos principais

Este módulo transforma a transcrição em candidatos fundamentados na base TOTVS e em até dez termos úteis para a revisão comercial. Quando o modelo e o índice local existem, usa E5; caso contrário, mantém BM25 com aliases como fallback auditável.


## Artefatos opcionais do E5

O índice `rag_multilingual_e5_small_embeddings.npz` é produzido pelo notebook 10 e não é versionado. O modelo e o índice são carregados somente na primeira consulta do modo `auto` ou `full`.

O arquivo de embeddings precisa corresponder aos IDs e à dimensão da base atual. Se estiver ausente, o modo `auto` passa a BM25; o modo `full` informa a falha.


In [ ]:
E5_MODEL_NAME = "intfloat/multilingual-e5-small"

E5_EMBEDDINGS_PATH = "data/processed/rag_multilingual_e5_small_embeddings.npz"

_E5_RETRIEVER = None

_E5_LOAD_ERROR = None

## Representação lexical

Campos mais específicos — produto, título e palavras-chave — recebem mais peso. Os aliases reconhecem como equivalentes formas como `Protheus` e `TOTVS Protheus`.

A mesma normalização prepara a consulta e os documentos, mas o peso dos campos permanece distinto. Isso favorece uma menção em nome de produto ou título em relação a uma palavra isolada no conteúdo.


In [ ]:
def _document_tokens(document: dict[str, Any]) -> list[str]:
    weighted_fields = [
        (document.get("product", ""), 4),
        (document.get("title", ""), 3),
        (" ".join(document.get("keywords", [])), 3),
        (" ".join(document.get("competitors", [])), 2),
        (document.get("category", ""), 1),
        (" ".join(document.get("segments", [])), 1),
        (" ".join(document.get("related_products", [])), 1),
        (document.get("content", ""), 1),
    ]
    return [token for text, weight in weighted_fields for token in _tokens(text) * weight]


### Consulta com aliases

A consulta combina palavras da transcrição com aliases da base TOTVS. Citações explícitas de produtos recebem reforço, preservando a distinção entre menção direta e semelhança contextual.


In [ ]:
def _query_tokens(transcription: str, alias_groups: list[dict[str, Any]]) -> tuple[list[str], set[str]]:
    normalized = f" {_normalize(transcription)} "
    query = _tokens(transcription)
    explicit_products: set[str] = set()
    for group in alias_groups:
        variants = [group["canonical"], *group.get("aliases", [])]
        if any(f" {_normalize(variant)} " in normalized for variant in variants):
            canonical = group["canonical"]
            explicit_products.add(_normalize(canonical))
            query.extend(_tokens(canonical) * 3)
    return query, explicit_products


## Evidências compartilhadas pelos rankings

BM25 e E5 calculam relevância de formas diferentes, mas entregam o mesmo contrato. Estas funções concentram a agregação das fontes e uma descrição curta de cada documento, evitando que os dois caminhos produzam estruturas divergentes.

Vários documentos podem sustentar o mesmo produto. O agrupamento conserva IDs, títulos e URLs únicas, enquanto a serialização apresenta até três candidatos em ordem de score.


In [ ]:
def _add_product_evidence(
    grouped: dict[str, dict[str, Any]],
    product: str,
    raw_score: float,
    explicit_match: bool,
    matched_terms: set[str],
    document: dict[str, Any],
    source_urls: list[str],
) -> None:
    candidate = grouped.setdefault(
        product,
        {
            "raw_score": float("-inf"),
            "explicit_match": False,
            "matched_terms": set(),
            "documents": {},
            "sources": [],
        },
    )
    candidate["raw_score"] = max(candidate["raw_score"], raw_score)
    candidate["explicit_match"] = candidate["explicit_match"] or explicit_match
    candidate["matched_terms"].update(matched_terms)
    candidate["documents"][document["id"]] = {
        "id": document["id"],
        "title": document.get("title"),
        "document_type": document.get("document_type"),
        "product": document.get("product"),
    }
    candidate["sources"].extend(source_urls)


### Serializar candidatos

Depois de reunir documentos por produto, esta etapa ordena os candidatos, limita a lista e monta o contrato de fontes. O score continua identificado pelo mecanismo que o produziu.


In [ ]:
def _serialize_product_candidates(
    grouped: dict[str, dict[str, Any]],
    top_k: int,
    *,
    score_type: str,
    engine: str,
    model: str | None,
    score_transform: Any,
) -> list[dict[str, Any]]:
    ranked = sorted(grouped.items(), key=lambda item: (-item[1]["raw_score"], item[0]))[:top_k]
    return [
        {
            "product": product,
            "score": round(min(max(score_transform(values["raw_score"]), 0.0), 1.0), 6),
            "score_type": score_type,
            "engine": engine,
            "model": model,
            "explicit_match": values["explicit_match"],
            "matched_terms": sorted(values["matched_terms"])[:10],
            "document_ids": list(values["documents"]),
            "documents": list(values["documents"].values()),
            "sources": list(dict.fromkeys(values["sources"])),
        }
        for product, values in ranked
    ]


## Ranking com fontes

O cálculo BM25 consolida documentos pelo produto, exige ao menos uma URL de fonte e conserva no máximo três candidatos. `explicit_match` distingue uma citação direta de uma correspondência apenas contextual.

BM25 serve como mecanismo local auditável. Seu score é normalizado relativamente ao melhor candidato nesta consulta, então não deve ser interpretado como chance de compra.


### Estatísticas do catálogo

Para cada documento, agrupe seus tokens, a frequência local e o comprimento em um registro nomeado. O índice também guarda o comprimento médio e a frequência global dos termos, usados pelo BM25 para reduzir o peso de palavras comuns.


In [ ]:
def _bm25_statistics(knowledge_base: list[dict[str, Any]]) -> dict[str, Any]:
    records = []
    document_frequency = Counter()
    for document in knowledge_base:
        tokens = _document_tokens(document)
        records.append({
            "document": document,
            "tokens": tokens,
            "term_frequency": Counter(tokens),
            "length": len(tokens),
        })
        document_frequency.update(set(tokens))
    average_length = sum(record["length"] for record in records) / max(len(records), 1)
    return {
        "records": records,
        "average_length": average_length,
        "document_frequency": document_frequency,
    }


### Score lexical por documento

A fórmula BM25 combina frequência na consulta, raridade no catálogo e ajuste pelo comprimento do documento. Ela produz uma medida de relevância, não de probabilidade.


In [ ]:
def _bm25_score(
    query: list[str], term_frequency: Counter, length: int,
    document_frequency: Counter, document_count: int, average_length: float,
) -> float:
    score = 0.0
    for term in query:
        frequency = term_frequency.get(term, 0)
        if not frequency:
            continue
        seen_in = document_frequency[term]
        inverse_document_frequency = math.log(
            1 + (document_count - seen_in + 0.5) / (seen_in + 0.5)
        )
        denominator = frequency + 1.5 * (1 - 0.75 + 0.75 * length / average_length)
        score += inverse_document_frequency * frequency * 2.5 / denominator
    return score


### Filtrar e agregar produtos

Uma menção explícita adiciona bônus ao score. Só documentos com score mínimo e URL de fonte entram no agrupamento; os candidatos finais herdam documentos e fontes verificáveis.


In [ ]:
def _rank_products_bm25(transcription: str, top_k: int = 3) -> list[dict[str, Any]]:
    knowledge_base, alias_groups = _load_catalog()
    query, explicit_products = _query_tokens(transcription, alias_groups)
    if not query:
        return []
    index = _bm25_statistics(knowledge_base)
    grouped: dict[str, dict[str, Any]] = {}
    for record in index["records"]:
        document = record["document"]
        tokens = record["tokens"]
        score = _bm25_score(
            query, record["term_frequency"], record["length"],
            index["document_frequency"], len(knowledge_base), index["average_length"],
        )
        product = document.get("product") or document.get("title")
        normalized_product = _normalize(product)
        explicit_match = any(
            alias in normalized_product
            or normalized_product in alias
            or set(_tokens(alias)).issubset(set(_tokens(product)))
            for alias in explicit_products
        )
        if explicit_match:
            score += 8.0
        if score < 2.0:
            continue
        source_urls = [
            source["url"] for source in document.get("sources", []) if source.get("url")
        ]
        if not source_urls:
            continue
        matched = set(query) & set(tokens)
        _add_product_evidence(
            grouped, product, score, explicit_match, matched, document, source_urls
        )
    if not grouped:
        return []
    highest_score = max(values["raw_score"] for values in grouped.values())
    return _serialize_product_candidates(
        grouped, top_k, score_type="heuristic", engine="bm25_aliases",
        model=None, score_transform=lambda raw_score: raw_score / highest_score,
    )


## Formato comum dos candidatos E5

A similaridade de cosseno é normalizada para o intervalo de zero a um somente para facilitar ordenação. Ela continua identificada como similaridade, não como probabilidade. Documentos sem fonte são descartados.

O E5 compara uma consulta com o índice de documentos. A transformação de cosseno para o intervalo de zero a um facilita a leitura, mas não calibra a medida para o negócio.


In [ ]:
def _format_e5_candidates(
    transcription: str, scores: list[float], top_k: int
) -> list[dict[str, Any]]:
    knowledge_base, alias_groups = _load_catalog()
    query, explicit_products = _query_tokens(transcription, alias_groups)
    grouped: dict[str, dict[str, Any]] = {}
    ranked_indices = sorted(range(len(scores)), key=lambda index: -scores[index])
    for index in ranked_indices:
        raw_score = float(scores[index])
        if raw_score < 0.20:
            continue
        document = knowledge_base[index]
        source_urls = [
            source["url"]
            for source in document.get("sources", [])
            if source.get("url")
        ]
        if not source_urls:
            continue
        product = document.get("product") or document.get("title")
        product_tokens = set(_tokens(product))
        explicit_match = any(
            set(_tokens(alias)).issubset(product_tokens) for alias in explicit_products
        )
        matched = set(query) & set(_document_tokens(document))
        _add_product_evidence(
            grouped, product, raw_score, explicit_match, matched, document, source_urls
        )

    return _serialize_product_candidates(
        grouped,
        top_k,
        score_type="normalized_cosine_similarity",
        engine="multilingual_e5_small",
        model=E5_MODEL_NAME,
        score_transform=lambda raw_score: (raw_score + 1.0) / 2.0,
    )


## Carregamento e roteamento do retriever

O loader confere IDs, quantidade e dimensão dos embeddings antes de criar o adaptador de consulta. `auto` registra o motivo do fallback; `full` exige E5; `fallback` usa BM25 diretamente.

As próximas células primeiro validam recursos, depois constroem a consulta e, por fim, selecionam o mecanismo conforme o modo. O carregamento só ocorre quando solicitado.


### Compatibilidade estrutural do índice

Um arquivo pode conter os IDs corretos e ainda estar truncado ou ter dimensão incompatível com o modelo. Essa validação ocorre no carregamento para que o modo `auto` possa recorrer ao BM25 de forma transparente.


In [ ]:
def _validate_e5_embeddings(
    embeddings: Any, expected_rows: int, expected_dimension: int | None = None
) -> None:
    if getattr(embeddings, "ndim", None) != 2:
        raise ValueError("Os embeddings E5 devem formar uma matriz bidimensional.")
    rows, dimension = embeddings.shape
    if rows != expected_rows:
        raise ValueError("A quantidade de embeddings E5 não corresponde à base TOTVS.")
    if expected_dimension is not None and dimension != expected_dimension:
        raise ValueError("A dimensão do índice E5 não corresponde ao modelo carregado.")


### Validar índice e carregar modelo

A primeira consulta carrega os recursos. Antes da inferência, compare IDs, número de linhas e dimensão com a base TOTVS e o encoder selecionado.


In [ ]:
def _load_e5_resources():
    import numpy as np
    import torch
    import torch.nn.functional as functional
    from transformers import AutoModel, AutoTokenizer

    embedding_path = _project_root() / E5_EMBEDDINGS_PATH
    if not embedding_path.is_file():
        raise FileNotFoundError(f"Índice E5 não encontrado: {embedding_path}")
    knowledge_base, _ = _load_catalog()
    saved = np.load(embedding_path, allow_pickle=False)
    expected_ids = [document["id"] for document in knowledge_base]
    if saved["document_ids"].tolist() != expected_ids:
        raise ValueError("O índice E5 não corresponde à versão atual da base TOTVS.")
    saved_embeddings = saved["embeddings"]
    _validate_e5_embeddings(saved_embeddings, len(expected_ids))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
    model = AutoModel.from_pretrained(E5_MODEL_NAME).to(device).eval()
    model_dimension = getattr(model.config, "hidden_size", None)
    if not isinstance(model_dimension, int):
        raise ValueError("O modelo E5 não informa a dimensão dos embeddings.")
    _validate_e5_embeddings(saved_embeddings, len(expected_ids), model_dimension)
    document_embeddings = torch.as_tensor(
        saved_embeddings, dtype=torch.float32, device=device
    )
    return torch, functional, device, tokenizer, model, document_embeddings


### Construir consulta E5 sob demanda

O texto recebe o prefixo `query:` usado pelo E5. O vetor médio é normalizado e comparado ao índice local; a função pronta permanece em cache no kernel.


In [ ]:
def _load_e5_retriever():
    global _E5_RETRIEVER, _E5_LOAD_ERROR
    if _E5_RETRIEVER is not None:
        return _E5_RETRIEVER
    if _E5_LOAD_ERROR is not None:
        raise RuntimeError("Retriever E5 indisponível.") from _E5_LOAD_ERROR
    try:
        torch, functional, device, tokenizer, model, document_embeddings = (
            _load_e5_resources()
        )

        def retrieve(text: str, top_k: int = 3) -> list[dict[str, Any]]:
            encoded = tokenizer(
                ["query: " + text], padding=True, truncation=True,
                max_length=512, return_tensors="pt",
            )
            encoded = {name: value.to(device) for name, value in encoded.items()}
            with torch.inference_mode():
                hidden = model(**encoded).last_hidden_state.float()
            mask = encoded["attention_mask"].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
            query_embedding = functional.normalize(pooled, p=2, dim=1)
            scores = (query_embedding @ document_embeddings.T)[0].cpu().tolist()
            return _format_e5_candidates(text, scores, top_k)

        _E5_RETRIEVER = retrieve
        return _E5_RETRIEVER
    except Exception as error:
        _E5_LOAD_ERROR = error
        raise RuntimeError("Retriever E5 indisponível.") from error


### Selecionar o mecanismo de produtos

`fallback` consulta BM25 sem carregar o E5. `auto` tenta E5 e registra uma causa segura quando precisa recorrer ao BM25; `full` exige modelo e índice compatíveis. A função devolve candidatos, mecanismo efetivo e eventual motivo do fallback.


In [ ]:
def _analyze_products(
    transcription: str, mode: str
) -> tuple[list[dict[str, Any]], str, dict[str, str] | None]:
    if mode == "fallback":
        return _rank_products_bm25(transcription), "fallback", None
    try:
        retriever = _load_e5_retriever()
    except Exception as error:
        if mode == "full":
            raise RuntimeError("O modo full exige o modelo e o índice E5.") from error
        return _rank_products_bm25(transcription), "fallback", _fallback_reason(error)
    return retriever(transcription, top_k=3), "model", None


## Proteção durante a extração de termos

E-mail, telefone e CPF são removidos somente da cópia usada para extrair termos. Essa filtragem não altera `transcricao_original`.

A remoção cobre padrões usuais, não é anonimização completa. Revise termos e fontes antes de compartilhar o resultado fora do ambiente autorizado.


In [ ]:
def _text_without_personal_data(text: str) -> str:
    sanitized = re.sub(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", " ", text, flags=re.I)
    sanitized = re.sub(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b", " ", sanitized)
    sanitized = re.sub(
        r"(?<!\d)(?:\+?55\s*)?(?:\(?\d{2}\)?[\s.-]*)?9?\d{4}[\s.-]*\d{4}(?!\d)",
        " ",
        sanitized,
    )
    return sanitized


## Priorização dos termos

Produtos citados vêm primeiro, seguidos de dores, concorrentes e vocabulário comercial. A frequência geral completa a lista somente quando ainda há espaço, sempre com limite de dez itens únicos.

A lista é descritiva e não altera o ranking de produtos. Cada termo aparece uma vez; o limite impede que frequência genérica oculte produtos e dores.


### Produtos citados

Encontre as variantes do catálogo no texto e devolva o nome canônico de cada produto, em ordem de primeira menção.


In [ ]:
def _matching_product_terms(normalized: str, alias_groups: list[dict[str, Any]]) -> list[str]:
    found: list[tuple[int, str]] = []
    for group in alias_groups:
        variants = [group["canonical"], *group.get("aliases", [])]
        positions = [
            normalized.find(_normalize(variant))
            for variant in variants
            if re.search(rf"(?<!\w){re.escape(_normalize(variant))}(?!\w)", normalized)
        ]
        if positions:
            found.append((min(positions), group["canonical"]))
    return [term for _, term in sorted(found)]


### Concorrentes documentados

Só entram nomes presentes na base de conhecimento e citados na transcrição. Uma ocorrência não equivale a comparação comercial confirmada.


In [ ]:
def _matching_competitors(normalized: str, knowledge_base: list[dict[str, Any]]) -> list[str]:
    competitors = {
        competitor
        for document in knowledge_base
        for competitor in document.get("competitors", [])
        if competitor
    }
    hits = [
        competitor for competitor in competitors
        if re.search(
            rf"(?<!\w){re.escape(_normalize(competitor))}(?!\w)", normalized
        )
    ]
    return sorted(hits, key=lambda item: (normalized.find(_normalize(item)), item))


### Vocabulário comercial

A lista reutiliza os sinais de intenção, compra e churn já apresentados nos outros módulos. Isso evita uma definição paralela de termos na saída integrada.


In [ ]:
def _matching_commercial_terms(normalized: str) -> list[str]:
    commercial_signals = (
        OPPORTUNITY_INTENT_SIGNALS
        | OPPORTUNITY_BUY_SIGNALS
        | set(HIGH_CHURN_SIGNALS)
        | set(MEDIUM_CHURN_SIGNALS)
    )
    hits = _signal_hits(normalized, commercial_signals)
    return sorted(hits, key=lambda item: (normalized.find(item), item))


### Consolidar até dez termos

Insira as categorias em ordem de prioridade e sem duplicatas. Palavras comuns por frequência completam a lista apenas quando ainda há espaço.


In [ ]:
def _extract_key_terms(transcription: str, limit: int = 10) -> list[str]:
    sanitized = _text_without_personal_data(transcription)
    normalized = _normalize(sanitized)
    knowledge_base, alias_groups = _load_catalog()
    prioritized: list[str] = []
    seen: set[str] = set()

    def add(term: str) -> None:
        canonical = _normalize(term).strip()
        if not canonical or canonical in seen or len(prioritized) >= limit:
            return
        prioritized.append(term)
        seen.add(canonical)

    for term in _matching_product_terms(normalized, alias_groups):
        add(term)
    pain_hits = _signal_hits(normalized, OPPORTUNITY_PAIN_SIGNALS)
    for term in sorted(pain_hits, key=lambda item: (normalized.find(item), item)):
        add(term)
    for term in _matching_competitors(normalized, knowledge_base):
        add(term)
    for term in _matching_commercial_terms(normalized):
        add(term)

    token_counts = Counter(_tokens(sanitized))
    first_position = {token: normalized.find(token) for token in token_counts}
    for token, _ in sorted(
        token_counts.items(),
        key=lambda item: (-item[1], first_position[item[0]], item[0]),
    ):
        if token.isdigit() or any(
            re.search(rf"(?<!\w){re.escape(token)}(?!\w)", existing)
            for existing in seen
        ):
            continue
        add(token)
    return prioritized
